# Lire des fichiers et contrôler la qualité des données

Atelier Python et Machine Learning pour la recherche scientifique · 12–16 octobre 2026

## Objectifs
- Lire un CSV avec la bibliothèque standard.
- Distinguer données absentes et invalides.
- Conserver une trace des problèmes.

**Mode d’emploi :** lire les explications, exécuter les cellules dans l’ordre avec `Maj + Entrée`, modifier les exemples puis réaliser les exercices. Les corrections sont dans le sous-dossier `solutions`.

Les exercices **Essentiel** constituent le parcours commun. Les exercices **Approfondissement** peuvent être terminés après la séance.

## 1. Retrouver les données
`Path` construit des chemins compatibles avec différents systèmes. Les données sont incluses dans le dossier `data` à la racine de l’atelier.

In [ ]:
from pathlib import Path
import pandas as pd

dossier = Path.cwd().resolve()
racine = next((p for p in [dossier, *dossier.parents] if (p / "data" / "batiments.csv").exists()), None)
if racine is None:
    raise FileNotFoundError("Décompressez le dossier complet et ouvrez ce notebook depuis ce dossier.")
df = pd.read_csv(racine / "data" / "batiments.csv")
df.head()

## 2. Lire un CSV sans pandas
`with` ferme le fichier à la fin du bloc. `DictReader` utilise les noms de colonnes comme clés. Les valeurs lues restent des chaînes de caractères.

In [ ]:
import csv

with (racine / "data" / "mesures_brutes.csv").open(encoding="utf-8", newline="") as fichier:
    lignes = list(csv.DictReader(fichier))
print(lignes)

## 3. Convertir et distinguer les problèmes
Une chaîne vide signale ici une mesure absente. Le mot `erreur` est une valeur invalide. Ne remplacez pas automatiquement ces valeurs par zéro : zéro peut être une mesure valide.

In [ ]:
mesures_valides = []
problemes = []
for ligne in lignes:
    texte = ligne["temperature_c"].strip()
    if texte == "":
        problemes.append({"id": ligne["echantillon"], "raison": "absente"})
        continue
    try:
        valeur = float(texte)
    except ValueError:
        problemes.append({"id": ligne["echantillon"], "raison": "non numérique"})
        continue
    mesures_valides.append({"echantillon": ligne["echantillon"], "temperature_c": valeur})
print(mesures_valides)
print(problemes)

## 4. Une fonction de contrôle réutilisable
Une borne physique dépend du domaine et du capteur. Les bornes ci-dessous sont un exemple pédagogique, pas une règle universelle. `math.isfinite` exclut également les valeurs infinies et NaN.

In [ ]:
import math

def temperature_valide(valeur, minimum=-50, maximum=60):
    return math.isfinite(valeur) and minimum <= valeur <= maximum

print(temperature_valide(25))
print(temperature_valide(999))

## 5. Exporter sans écraser les données brutes
Conservez les originaux et enregistrez les résultats dans un dossier distinct. Documentez le nombre de lignes conservées et exclues.

In [ ]:
sorties = racine / "resultats"
sorties.mkdir(exist_ok=True)
with (sorties / "mesures_valides.csv").open("w", encoding="utf-8", newline="") as fichier:
    auteur = csv.DictWriter(fichier, fieldnames=["echantillon", "temperature_c"])
    auteur.writeheader()
    auteur.writerows(mesures_valides)
print("Conservées :", len(mesures_valides), "Problèmes :", len(problemes))

## Exercices progressifs sur les fichiers — Corrigé

Ces exercices sont adaptés de *The Python Workbook*.


### Exercice 1 — Afficher le début d’un fichier


In [ ]:
from pathlib import Path

def afficher_debut(chemin, n=5):
    try:
        with Path(chemin).open(encoding="utf-8") as fichier:
            for indice, ligne in enumerate(fichier):
                if indice >= n:
                    break
                print(ligne.rstrip())
    except FileNotFoundError:
        print("Fichier introuvable.")

afficher_debut(racine / "data" / "mesures_brutes.csv")


### Exercice 2 — Afficher la fin d’un fichier


In [ ]:
def afficher_fin(chemin, n=5):
    try:
        with Path(chemin).open(encoding="utf-8") as fichier:
            lignes = fichier.readlines()

        for ligne in lignes[-n:]:
            print(ligne.rstrip())
    except FileNotFoundError:
        print("Fichier introuvable.")

afficher_fin(racine / "data" / "mesures_brutes.csv")


### Exercice 3 — Convertir des valeurs en gérant les erreurs


In [ ]:
def convertir_nombre(texte):
    texte = texte.strip()

    if texte == "":
        return None

    try:
        return float(texte)
    except ValueError:
        return None

textes = ["12.5", "", "erreur", "-3", "18.2"]
converties = [convertir_nombre(texte) for texte in textes]
valides = [valeur for valeur in converties if valeur is not None]

print(converties)
print(valides)


### Exercice 4 — Défi : somme de valeurs stockées dans un fichier


In [ ]:
fichier_exemple = sorties / "valeurs_exercice.txt"

with fichier_exemple.open("w", encoding="utf-8") as fichier:
    fichier.write("12.5\n")
    fichier.write("7.2\n")
    fichier.write("erreur\n")
    fichier.write("-1.0\n")

total = 0
ignorees = 0

with fichier_exemple.open(encoding="utf-8") as fichier:
    for ligne in fichier:
        try:
            total += float(ligne.strip())
        except ValueError:
            ignorees += 1

print("Somme :", total)
print("Lignes ignorées :", ignorees)
